Model

In [1]:
import anndata
import numpy as np
import pandas as pd
from scipy.sparse import spmatrix
import scanpy as sc
from sklearn.preprocessing import StandardScaler

import celltypist
from celltypist import models

from models.classifier import Classifier

import logging
logger = logging.getLogger("Data-Preprocessing")

Data Preprocessing

In [2]:
models.models_path = "/home/lagandua/projects/sven/celltypist/.cache"
ct_model = models.Model.load(model = 'Immune_All_Low.pkl')
label = "Predicted_labels_CellTypist"

data_path = "./data/global.h5ad"
raw = anndata.read_h5ad(data_path)
logger.info("🍳 Preparing data before training")
raw.var_names_make_unique()
sc.pp.normalize_total(raw, target_sum=1e4)
sc.pp.log1p(raw)
indata = raw.X
genes = raw.var_names
labels = raw.obs[label]

if isinstance(indata, pd.DataFrame):
    indata = indata.values
elif isinstance(indata, spmatrix):
    indata = indata.toarray()

labels = np.array(labels)
genes = np.array(genes)

# Check
if np.abs(np.expm1(indata[0]).sum()-10000) > 1:
    raise ValueError(
            "🛑 Invalid expression matrix, expect log1p normalized expression to 10000 counts per cell")
if len(labels) != indata.shape[0]:
    raise ValueError(
            f"🛑 Length of training labels ({len(labels)}) does not match the number of input cells ({indata.shape[0]})")
if len(genes) != indata.shape[1]:
    raise ValueError(
            f"🛑 The number of genes ({len(genes)}) provided does not match the number of genes in the training data ({indata.shape[1]})")

# Filter
flag = indata.sum(axis = 0) == 0
if flag.sum() > 0:
    logger.info(f"✂️ {flag.sum()} non-expressed are filtered out")
    chosen_genes = np.array(list(set(ct_model.classifier.features).intersection(set(genes))))
    keep = (~flag) & (np.isin(genes, chosen_genes))
    genes = genes[keep]
logger.info(f"🔬 Input data has {indata.shape[0]} cells and {(keep).sum()} genes")


🍳 Preparing data before training
✂️ 3912 non-expressed are filtered out
🔬 Input data has 329762 cells and 6143 genes


In [3]:
# Scaler (this takes forever to run)
logger.info(f"⚖️ Scaling input data")
scaler = StandardScaler(with_mean = True)
indata = scaler.fit_transform(indata[:, keep] if keep.sum() > 0 else indata)
indata[indata > 10] = 10

⚖️ Scaling input data


In [6]:

# Save preprocessed data into train and test sets (this also takes forever)
logger.info(f"📝 Saving preprocessed data into train and validation sets")

val_size = 0.1 # 10%
true_count = int(indata.shape[0] * val_size)
false_count = indata.shape[0] - true_count

val_flag = np.concatenate((np.full(true_count, True), np.full(false_count, False)))
np.random.shuffle(val_flag)

np.savez('./data/train.npz', data=indata[~val_flag], labels=labels[~val_flag], genes=genes)
np.savez('./data/val.npz', data=indata[val_flag], labels=labels[val_flag], genes=genes)

📝 Saving preprocessed data into train and validation sets
